## Setup & Environment Detection

In [1]:
import sys

IS_COLAB = 'google.colab' in sys.modules

In [2]:
import os
import sys

if IS_COLAB:
    print("Running in Google Colab environment.")
    if os.path.exists('/content/aai521_3proj'):
        print("Repository already exists. Pulling latest changes...")
        %cd /content/aai521_3proj
        !git pull
    else:
        print("Cloning repository...")
        !git clone https://github.com/swapnilprakashpatil/aai521_3proj.git
        %cd aai521_3proj    
    %pip install -r requirements.txt --quiet
    sys.path.append('/content/aai521_3proj/src')
else:
    print("Running in local environment.")
    sys.path.append('../src')

Running in local environment.


## 1. Imports & Configuration

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
import config
from dataset import FloodDataset, create_dataloaders
from models import create_model
from metrics import SegmentationMetrics
from gpu_manager import GPUManager
from visualizations import (
    plot_confusion_matrix,
    visualize_predictions,
    compute_and_print_metrics
)
from evaluate import (
    run_inference,
    compute_sample_iou,
    analyze_error_distribution,
    save_evaluation_results,
    export_predictions,
    print_evaluation_summary
)
# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Initialize GPU
gpu_mgr = GPUManager()
gpu_mgr.setup()
device = gpu_mgr.get_device()

print(f"Device: {device}")

Device: cpu


## 2. Model Selection & Loading

We'll select the best model based on validation performance from the training results.

In [4]:
AVAILABLE_MODELS = {
    'fc_siam_diff': 'FC-Siam-Diff (Change Detection)',
    'siamese_unet++': 'Siamese U-Net++',
    'deeplabv3+': 'DeepLabV3+',
    'unet++': 'U-Net++',
    'segformer': 'SegFormer',
    'stanet': 'STANet'
}

# Select best model based on validation IoU: SegFormer (0.7755)
SELECTED_MODEL = 'segformer'

print(f"Selected Model: {AVAILABLE_MODELS[SELECTED_MODEL]}")
print(f"Note: SegFormer achieved the highest validation IoU (0.7755) among all models")

Selected Model: SegFormer
Note: SegFormer achieved the highest validation IoU (0.7755) among all models


In [5]:
# Find the best checkpoint for the selected model
training_dir = Path('../outputs/training')

model_dirs = []
if training_dir.exists():
    for d in training_dir.iterdir():
        if d.is_dir() and SELECTED_MODEL in d.name.lower():
            model_dirs.append(d)

if not model_dirs:
    print(f"ERROR: No training checkpoints found for {SELECTED_MODEL}")
else:
    latest_dir = sorted(model_dirs, key=lambda x: x.name)[-1]
    checkpoint_path = latest_dir / 'checkpoints' / 'best_model.pth'
    
    if checkpoint_path.exists():
        print(f"Found checkpoint: {checkpoint_path}")
    else:
        print(f"ERROR: No best_model.pth found in {latest_dir / 'checkpoints'}")
        checkpoint_path = None

Found checkpoint: ..\outputs\training\segformer_20251206_223120\checkpoints\best_model.pth


In [6]:
# Load the model
if checkpoint_path and checkpoint_path.exists():
    model = create_model(
        model_name=SELECTED_MODEL,
        in_channels=6 if 'siamese' not in SELECTED_MODEL.lower() else 3,
        num_classes=config.NUM_CLASSES,
        **config.MODEL_CONFIGS.get(SELECTED_MODEL, {})
    )
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
else:
    print("WARNING: No checkpoint available")
    model = None

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([7]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([7, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SegFormer initialized:
  Model: nvidia/segformer-b0-finetuned-ade-512-512
  Input channels: 6
  Output classes: 7
  Pretrained: True
Model loaded: 3,720,647 parameters


## 3. Load Test Data

In [ ]:
# Create test dataloader with memory-efficient settings
# Note: 3573 test samples - we'll use small batch size and avoid storing all images
_, _, test_loader = create_dataloaders(
    train_dir=config.PROCESSED_TRAIN_DIR,
    val_dir=config.PROCESSED_VAL_DIR,
    test_dir=config.PROCESSED_TEST_DIR,
    batch_size=2,  # Small batch size for stability
    num_workers=0,  # No multiprocessing
    pin_memory=False
)

print(f"Test set: {len(test_loader.dataset)} samples")
print(f"Batch size: 2 (memory-efficient for large dataset)")
print(f"Note: Only storing subset of images for visualization to prevent memory issues")

Loaded train dataset: 8346 samples

Class distribution (train):
  Class 0: 2,140,375,152 pixels (97.83%), weight: 0.0838
  Class 1: 32,315,521 pixels (1.48%), weight: 0.6821
  Class 2: 6,455,152 pixels (0.30%), weight: 1.5262
  Class 3: 0 pixels (0.00%), weight: 0.0829
  Class 4: 0 pixels (0.00%), weight: 0.0829
  Class 5: 1,574,657 pixels (0.07%), weight: 3.0902
  Class 6: 7,133,342 pixels (0.33%), weight: 1.4519
Loaded val dataset: 1199 samples

Class distribution (val):
  Class 0: 306,006,674 pixels (97.36%), weight: 0.0998
  Class 1: 5,155,887 pixels (1.64%), weight: 0.7687
  Class 2: 1,714,566 pixels (0.55%), weight: 1.3330
  Class 3: 0 pixels (0.00%), weight: 0.0985
  Class 4: 0 pixels (0.00%), weight: 0.0985
  Class 5: 357,567 pixels (0.11%), weight: 2.9189
  Class 6: 1,075,962 pixels (0.34%), weight: 1.6827
Loaded test dataset: 3573 samples

Class distribution (test):
  Class 0: 936,640,512 pixels (100.00%), weight: 1.0000
  Class 1: 0 pixels (0.00%), weight: 1.0000
  Class 2: 

: 

## 4. Inference on Test Set

In [ ]:
if model is not None:
    # Run memory-efficient inference
    # Note: Only stores first 100 images to prevent memory issues with 3573 samples
    print("Starting inference (this may take a few minutes)...")
    test_images, test_predictions, test_targets = run_inference(
        model, test_loader, device, store_images=True
    )
    print(f"\nInference complete!")
    print(f"Predictions shape: {test_predictions.shape}")
    print(f"Stored images for visualization: {test_images.shape[0]}")
else:
    print("WARNING: No model loaded")

Starting inference (this may take a few minutes)...


Running inference:  95%|█████████▌| 1701/1787 [20:11<00:59,  1.44it/s]

## 5. Quantitative Evaluation

In [ ]:
if model is not None:
    test_metrics, metrics_df = compute_and_print_metrics(
        test_predictions,
        test_targets,
        config.NUM_CLASSES,
        config.CLASS_NAMES
    )
else:
    print("WARNING: No predictions available")

## 6. Qualitative Visualization & Best/Worst Predictions

In [ ]:
if model is not None:
    num_viz = min(5, len(test_predictions))
    random_indices = np.random.choice(len(test_predictions), num_viz, replace=False)
    
    visualize_predictions(
        test_images,
        test_predictions,
        test_targets,
        random_indices,
        config.CLASS_NAMES
    )
else:
    print("WARNING: No predictions available")

In [ ]:
if model is not None:
    # Compute per-sample IoU
    sample_ious = compute_sample_iou(test_predictions, test_targets)
    
    # Find best and worst samples
    best_indices = np.argsort(sample_ious)[-3:][::-1]
    worst_indices = np.argsort(sample_ious)[:3]
    
    print("Best Predictions (Highest IoU):")
    for rank, idx in enumerate(best_indices, 1):
        print(f"  {rank}. Sample {idx}: IoU = {sample_ious[idx]:.4f}")
    
    print("\nWorst Predictions (Lowest IoU):")
    for rank, idx in enumerate(worst_indices, 1):
        print(f"  {rank}. Sample {idx}: IoU = {sample_ious[idx]:.4f}")
    
    # Visualize
    visualize_predictions(test_images, test_predictions, test_targets, best_indices, config.CLASS_NAMES)
    visualize_predictions(test_images, test_predictions, test_targets, worst_indices, config.CLASS_NAMES)
else:
    print("WARNING: No predictions available")

## 7. Error Analysis

In [ ]:
if model is not None:
    error_stats = analyze_error_distribution(test_predictions, test_targets, config.CLASS_NAMES)
    
    print("Class Distribution:")
    for class_name, stats in error_stats['class_distribution'].items():
        print(f"{class_name}: GT={stats['ground_truth_count']:,} ({stats['ground_truth_pct']:.2f}%), "
              f"Pred={stats['predicted_count']:,} ({stats['predicted_pct']:.2f}%)")
    
    if error_stats['misclassifications']:
        print("\nCommon Misclassifications:")
        for true_class, misclass in error_stats['misclassifications'].items():
            for pred_class, stats in misclass.items():
                print(f"  {true_class} → {pred_class}: {stats['percentage']:.2f}%")
else:
    print("WARNING: No predictions available")

## 8. Save Results

In [ ]:
if model is not None:
    results_dir = Path('../outputs/final_results')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    results_file, csv_file = save_evaluation_results(
        model_name=SELECTED_MODEL,
        checkpoint_path=checkpoint_path,
        test_metrics=test_metrics,
        metrics_df=metrics_df,
        sample_ious=sample_ious,
        output_dir=results_dir,
        timestamp=timestamp
    )
    
    print(f"Results saved to: {results_file}")
else:
    print("WARNING: No results to save")

## 9. Summary Report

In [ ]:
if model is not None:
    print_evaluation_summary(
        model_name=AVAILABLE_MODELS[SELECTED_MODEL],
        model_params=sum(p.numel() for p in model.parameters()),
        test_size=len(test_loader.dataset),
        test_metrics=test_metrics,
        sample_ious=sample_ious,
        class_names=config.CLASS_NAMES
    )
else:
    print("WARNING: No model evaluated")

## 10. Export Results

In [ ]:
if model is not None:
    submission_dir = Path('../outputs/submission')
    
    predictions_file, metadata_file = export_predictions(
        model_name=SELECTED_MODEL,
        predictions=test_predictions,
        test_metrics=test_metrics,
        class_names=config.CLASS_NAMES,
        output_dir=submission_dir,
        timestamp=timestamp
    )
    
    print(f"Predictions saved: {predictions_file} ({predictions_file.stat().st_size / 1e6:.2f} MB)")
else:
    print("WARNING: No predictions to export")

## 13. Next Steps & Future Work

### Immediate Actions:
1. **Model Ensemble**: Combine predictions from top 3 models
2. **Post-Processing**: Apply CRF or morphological operations
3. **Test-Time Augmentation**: Flip/rotate images during inference

### Future Improvements:
1. **Data Augmentation**: More aggressive augmentation for minority classes
2. **Semi-Supervised Learning**: Use unlabeled flood images
3. **Active Learning**: Identify and label hard examples
4. **Multi-Task Learning**: Predict flood depth/severity simultaneously

### Deployment Considerations:
1. **Model Optimization**: Quantization, pruning, ONNX export
2. **API Development**: REST API for inference service
3. **Monitoring**: Track prediction quality in production
4. **Retraining Pipeline**: Automated retraining with new data

---

**Thank you for using this flood detection system!**